# BPLLM trên Kaggle (2/2): server LLM chuẩn OpenAI cho Streamlit

Nạp **3 model fine-tune** (Food Delivery, Food Delivery + E-commerce, Food Delivery +
Reimbursement), mở API `/v1/chat/completions` giống OpenAI, rồi đưa ra internet qua
Cloudflare Tunnel. Streamlit trên máy bạn gọi tới URL này; trường `model` của request
chọn model nào trả lời. Không gửi gì cho OpenAI.

**Vì sao nén 4-bit khi nạp:** mỗi model F16 nặng 16 GB, 2 GPU T4 chỉ chứa được 1 model.
Nén 4-bit (bitsandbytes NF4) còn khoảng 5,5 GB mỗi model, nên chứa đủ 3. Câu trả lời có thể
hơi khác bản F16, vì vậy **số liệu thực nghiệm lấy từ notebook 1/2**, không lấy từ server này.

**Cài đặt ở cột bên phải:** Accelerator **GPU T4 x2**, Internet **On**.
Nên đặt API key trong **Add-ons → Secrets**, tên `BPLLM_API_KEY` (cùng giá trị `LLM_API_KEY` trong `.env`).

Chạy lần lượt từ trên xuống, giữ tab notebook mở trong lúc dùng app. Phiên Kaggle tối đa
12 giờ; mỗi lần chạy lại, URL tunnel đổi, cần sửa `LLM_BASE_URL` trong `.env`.

## Khối 1. Cấu hình

In [ ]:
QUANT = "int4"          # bản model (tên model trên Hugging Face); app mặc định gọi int4
LOAD_IN_4BIT = True     # bắt buộc True để 3 model vừa 2 GPU T4
PORT = 8000

# Bản chép của src/bpllm/experiments.py
MODELS = {
    "food_delivery": {"int4": "angeloc1/llama3dot1FoodDel4v05",
                      "int8": "angeloc1/llama3dot1FoodDel8v02"},
    "similar": {"int4": "angeloc1/llama3dot1SimilarProcesses4",
                "int8": "angeloc1/llama3dot1SimilarProcesses8"},
    "different": {"int4": "angeloc1/llama3dot1DifferentProcesses4",
                  "int8": "angeloc1/llama3dot1DifferentProcesses8"},
}
# Model nào nằm trên GPU nào (mỗi model 4-bit ~5,5 GB, mỗi T4 ~15 GB)
DEVICES = {"food_delivery": "cuda:0", "similar": "cuda:1", "different": "cuda:0"}

API_KEY = None
try:
  from kaggle_secrets import UserSecretsClient
  API_KEY = UserSecretsClient().get_secret("BPLLM_API_KEY")
except Exception:
  pass
API_KEY = API_KEY or "change-me"   # đổi giá trị này nếu không dùng Kaggle Secrets
assert API_KEY != "change-me", "Hãy đặt API key riêng (Kaggle Secret BPLLM_API_KEY hoặc sửa dòng trên)"
SERVED = {group: models[QUANT] for group, models in MODELS.items()}
print("Sẽ nạp:", SERVED)

In [ ]:
!nvidia-smi

## Khối 2. Cài thư viện

Chỉ cài gói còn thiếu; không hạ cấp thư viện Kaggle cài sẵn.

In [ ]:
import importlib.metadata as md
import subprocess
import sys

from packaging.version import Version

MISSING = []


def need(package, minimum):
  try:
    current = md.version(package)
  except md.PackageNotFoundError:
    current = None
  if current is None or (minimum and Version(current) < Version(minimum)):
    MISSING.append(f"{package}>={minimum}" if minimum else package)
  return current


need("transformers", "4.45")
need("accelerate", "0.33")
need("bitsandbytes", "0.43")
need("fastapi", None)
need("uvicorn", None)

if MISSING:
  print("Cài:", MISSING)
  subprocess.run([sys.executable, "-m", "pip", "install", "-q", *MISSING], check=True)
  print("Đã cài xong. Nếu vừa nâng cấp transformers: Run → Restart & clear cell outputs, rồi chạy lại từ Khối 1.")
for package in ("torch", "transformers", "huggingface_hub", "accelerate", "bitsandbytes"):
  try:
    print(f"{package:16} {md.version(package)}")
  except md.PackageNotFoundError:
    print(f"{package:16} (chưa có)")

## Khối 3a. Chuẩn bị tải model

Đặt HF token (không bắt buộc) ở **Add-ons → Secrets**, tên `HF_TOKEN` (token đọc của
Hugging Face), để tránh bị giới hạn khi tải ẩn danh 3 × 16 GB.

In [ ]:
import os
import shutil
import time

import torch
import transformers
from huggingface_hub import snapshot_download
from packaging.version import Version

HF_TOKEN = None
try:
  from kaggle_secrets import UserSecretsClient
  HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
  pass
print("HF token:", "có (Kaggle Secret HF_TOKEN)" if HF_TOKEN else "không (tải ẩn danh, có thể bị giới hạn tốc độ)")

# transformers >= 4.56 đổi torch_dtype thành dtype
DTYPE_KW = ({"dtype": torch.float16} if Version(transformers.__version__) >= Version("4.56")
            else {"torch_dtype": torch.float16})


def free_gb(path):
  return shutil.disk_usage(path).free / 2**30


# Thư mục tải model: chọn ổ còn nhiều chỗ nhất (mỗi model cần khoảng 16,1 GB)
MODEL_ROOT = max(["/tmp", "/kaggle/working"], key=free_gb)
MODEL_ROOT = os.path.join(MODEL_ROOT, "bpllm_models")
os.makedirs(MODEL_ROOT, exist_ok=True)
print(f"Tải model vào {MODEL_ROOT}, còn trống {free_gb(MODEL_ROOT):.1f} GB")


def download(model_id):
  """Tải trọng số + tokenizer (bỏ training_args.bin); trả về thư mục local. Lỗi thì nói rõ nguyên nhân."""
  target = os.path.join(MODEL_ROOT, model_id.replace("/", "__"))
  if free_gb(MODEL_ROOT) < 17:
    raise RuntimeError(f"Ổ {MODEL_ROOT} chỉ còn {free_gb(MODEL_ROOT):.1f} GB, cần ~17 GB. "
                       "Xóa model cũ trong thư mục này hoặc Restart session.")
  start = time.time()
  for attempt in range(1, 4):
    try:
      path = snapshot_download(model_id, local_dir=target, token=HF_TOKEN, max_workers=8,
                               allow_patterns=["*.json", "*.safetensors"])
      break
    except Exception as error:
      print(f"  Lần {attempt} tải {model_id} lỗi: {type(error).__name__}: {error}")
      if attempt == 3:
        raise RuntimeError(
            "Không tải được model. Kiểm tra: (1) Settings → Internet: On; "
            "(2) đã xác minh số điện thoại Kaggle; (3) thêm Kaggle Secret HF_TOKEN nếu lỗi 429/401; "
            "(4) dung lượng ổ ở dòng in phía trên.") from error
      time.sleep(15 * attempt)
  shards = [f for f in os.listdir(path) if f.endswith(".safetensors")]
  print(f"  Đã tải {model_id}: {len(shards)} file safetensors, {time.time() - start:.0f} s")
  return path


def remove_download(path):
  shutil.rmtree(path, ignore_errors=True)

## Khối 3b. Tải và nạp 3 model

Lần lượt từng model: tải ~16 GB, nén 4-bit lên GPU, xóa bản tải để ổ đĩa không đầy.
Mất khoảng 15 đến 25 phút. Chạy lại khối này thì model đã nạp được bỏ qua.

In [ ]:
import gc
import threading

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

ENGINES = globals().get("ENGINES", {})   # model id -> (tokenizer, model, lock)

for group, model_id in SERVED.items():
  if model_id in ENGINES:
    print("Đã có:", model_id)
    continue
  print("Đang tải", model_id, "→", DEVICES[group])
  path = download(model_id)
  quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16) if LOAD_IN_4BIT else None
  tok = AutoTokenizer.from_pretrained(path)
  mdl = AutoModelForCausalLM.from_pretrained(path, quantization_config=quant,
                                             device_map={"": DEVICES[group]}, **DTYPE_KW).eval()
  ENGINES[model_id] = (tok, mdl, threading.Lock())
  remove_download(path)
  gc.collect()
  torch.cuda.empty_cache()
  print("  xong; GPU đã dùng: " + ", ".join(
      f"cuda:{i} {torch.cuda.memory_allocated(i) / 2**30:.1f} GB" for i in range(torch.cuda.device_count())))

## Khối 4. Server chuẩn OpenAI (`/v1/models`, `/v1/chat/completions`)

In [ ]:
import time
import uuid
from typing import Dict, List, Optional

import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel


class Message(BaseModel):
  role: str
  content: str


class ChatRequest(BaseModel):
  model: str
  messages: List[Message]
  max_tokens: Optional[int] = 256
  temperature: Optional[float] = 0.0


app = FastAPI(title="BPLLM LLM server")


def check_key(authorization):
  if authorization != f"Bearer {API_KEY}":
    raise HTTPException(status_code=401, detail="Sai API key")


def generate(model_id, messages, max_tokens):
  tok, mdl, lock = ENGINES[model_id]
  inputs = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                   return_dict=True, return_tensors="pt").to(mdl.device)
  with lock, torch.no_grad():
    out = mdl.generate(**inputs, max_new_tokens=max_tokens, do_sample=False,
                       temperature=None, top_p=None, pad_token_id=tok.eos_token_id)
  new_tokens = out[0][inputs["input_ids"].shape[-1]:]
  return tok.decode(new_tokens, skip_special_tokens=True).strip(), inputs["input_ids"].shape[-1], len(new_tokens)


@app.get("/v1/models")
def list_models(authorization: str = Header(default="")):
  check_key(authorization)
  return {"object": "list", "data": [{"id": m, "object": "model", "owned_by": "angeloc1"}
                                     for m in ENGINES]}


@app.post("/v1/chat/completions")
def chat(req: ChatRequest, authorization: str = Header(default="")):
  check_key(authorization)
  if req.model not in ENGINES:
    raise HTTPException(status_code=404, detail=f"Model {req.model} chưa được nạp: {list(ENGINES)}")
  text, n_in, n_out = generate(req.model, [m.model_dump() for m in req.messages],
                               min(req.max_tokens or 256, 1024))
  return {"id": f"chatcmpl-{uuid.uuid4().hex}", "object": "chat.completion",
          "created": int(time.time()), "model": req.model,
          "choices": [{"index": 0, "message": {"role": "assistant", "content": text},
                       "finish_reason": "stop"}],
          "usage": {"prompt_tokens": n_in, "completion_tokens": n_out,
                    "total_tokens": n_in + n_out}}


SERVER = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=SERVER.run, daemon=True).start()
time.sleep(3)
print("Server chạy ở cổng", PORT)

## Khối 5. Tự kiểm tra server trong Kaggle

In [ ]:
import requests

headers = {"Authorization": f"Bearer {API_KEY}"}
print(requests.get(f"http://localhost:{PORT}/v1/models", headers=headers, timeout=30).json())
reply = requests.post(f"http://localhost:{PORT}/v1/chat/completions", headers=headers, timeout=300, json={
    "model": SERVED["food_delivery"], "max_tokens": 32,
    "messages": [{"role": "user", "content": "Answer 'yes' or 'no': is Food Delivery a business process?"}]})
print(reply.status_code, reply.json()["choices"][0]["message"]["content"])

## Khối 6. Mở ra internet (tunnel)

1. Thử **Cloudflare quick tunnel** (không cần tài khoản), dùng giao thức `http2` vì mạng Kaggle
   thường chặn QUIC/UDP mà cloudflared dùng mặc định. Thử tối đa 3 lần, in log nếu thất bại.
2. Nếu không được và có Kaggle Secret `NGROK_AUTHTOKEN` (tài khoản ngrok miễn phí,
   <https://dashboard.ngrok.com/get-started/your-authtoken>), chuyển sang **ngrok**.

Khối chỉ in `LLM_BASE_URL=...` sau khi đã gọi thử được `/v1/models` qua link công khai;
không có link thì báo lỗi, không in `None`.

In [ ]:
import os
import re
import subprocess
import sys
import threading
import time
import urllib.request

import requests

CLOUDFLARED = "/kaggle/working/cloudflared"
CLOUDFLARED_URL = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
                   "cloudflared-linux-amd64")
TUNNEL_LOG = []


def public_ok(url, wait=90):
  """Gọi thử /v1/models qua link công khai (DNS của link mới cần vài chục giây)."""
  deadline = time.time() + wait
  while time.time() < deadline:
    try:
      r = requests.get(f"{url}/v1/models", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=15)
      if r.status_code == 200:
        return True
      TUNNEL_LOG.append(f"[thử link] HTTP {r.status_code}")
    except Exception as error:
      TUNNEL_LOG.append(f"[thử link] {type(error).__name__}")
    time.sleep(5)
  return False


def start_cloudflared(timeout=90):
  if not os.path.exists(CLOUDFLARED) or os.path.getsize(CLOUDFLARED) < 10_000_000:
    urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED)
  os.chmod(CLOUDFLARED, 0o755)
  proc = subprocess.Popen([CLOUDFLARED, "tunnel", "--no-autoupdate", "--protocol", "http2",
                           "--url", f"http://localhost:{PORT}"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
  found = {}

  def read_log():
    for line in proc.stdout:
      TUNNEL_LOG.append(line.rstrip())
      match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
      if match and "url" not in found:
        found["url"] = match.group(0)

  threading.Thread(target=read_log, daemon=True).start()
  deadline = time.time() + timeout
  while time.time() < deadline and "url" not in found and proc.poll() is None:
    time.sleep(1)
  return proc, found.get("url")


def start_ngrok(token):
  subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
  from pyngrok import conf, ngrok
  conf.get_default().auth_token = token
  return ngrok.connect(PORT, "http").public_url


TUNNEL, TUNNEL_KIND, PUBLIC_URL = None, None, None
if globals().get("SERVER") is None or not SERVER.started:
  raise RuntimeError("Server ở Khối 4 chưa chạy: chạy Khối 4 và 5 trước.")

for attempt in range(1, 4):
  proc, url = start_cloudflared()
  if url and public_ok(url):
    TUNNEL, TUNNEL_KIND, PUBLIC_URL = proc, "cloudflare", url
    break
  proc.kill()
  print(f"Cloudflare lần {attempt} không được ({'có link nhưng không gọi được' if url else 'không có link'}).")
  time.sleep(5)

if PUBLIC_URL is None:
  NGROK_TOKEN = None
  try:
    from kaggle_secrets import UserSecretsClient
    NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
  except Exception:
    pass
  if NGROK_TOKEN:
    url = start_ngrok(NGROK_TOKEN)
    if public_ok(url):
      TUNNEL_KIND, PUBLIC_URL = "ngrok", url


def tunnel_alive():
  if TUNNEL_KIND == "cloudflare":
    return TUNNEL.poll() is None
  if TUNNEL_KIND == "ngrok":
    from pyngrok import ngrok
    return any(t.public_url == PUBLIC_URL for t in ngrok.get_tunnels())
  return False


if PUBLIC_URL is None:
  print("\n".join(TUNNEL_LOG[-40:]))
  raise RuntimeError(
      "Không mở được link công khai. Xem log phía trên. Cách xử lý: thêm Kaggle Secret "
      "NGROK_AUTHTOKEN (tài khoản ngrok miễn phí) rồi chạy lại khối này.")

print(f"Tunnel: {TUNNEL_KIND}, đã gọi thử /v1/models qua link công khai: OK")
print("Chép 2 dòng sau vào .env trên máy bạn:")
print(f"LLM_BASE_URL={PUBLIC_URL}/v1")
print("LLM_API_KEY=<giá trị BPLLM_API_KEY>")

## Khối 7. Giữ phiên chạy

Khối này in trạng thái mỗi 5 phút. Để nó chạy và giữ tab mở trong lúc dùng Streamlit.
Muốn dừng: bấm Stop (■), server và tunnel tắt theo phiên.

In [ ]:
while True:
  alive = tunnel_alive()
  used = ", ".join(f"cuda:{i} {torch.cuda.memory_allocated(i) / 2**30:.1f} GB"
                   for i in range(torch.cuda.device_count()))
  print(time.strftime("%H:%M:%S"), "| tunnel", TUNNEL_KIND, "OK" if alive else "ĐÃ TẮT", "|",
        PUBLIC_URL, "|", used)
  if not alive:
    print("Tunnel đã tắt: chạy lại Khối 6 để mở link mới (link sẽ đổi).")
    break
  time.sleep(300)